In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import glob
import os
import pandas as pd
import multiprocessing
import logging
import blimpy as bl
%matplotlib inline

In [2]:
df = pd.read_csv('/datax/scratch/benjb/bl_nearby_stars/BL_NSS_cadences_turboSETI_BLISS_063026.csv')

In [3]:
bd1 = df['bliss .dat path 1'].values
bd2 = df['bliss .dat path 2'].values
bd3 = df['bliss .dat path 3'].values
bd4 = df['bliss .dat path 4'].values
bd5 = df['bliss .dat path 5'].values
bd6 = df['bliss .dat path 6'].values

h51 = df['.h5 path 1'].values
h52 = df['.h5 path 2'].values
h53 = df['.h5 path 3'].values
h54 = df['.h5 path 4'].values
h55 = df['.h5 path 5'].values
h56 = df['.h5 path 6'].values

# these are all the same
i1 = df['Unnamed: 0'].values
i2 = df['Unnamed: 0'].values
i3 = df['Unnamed: 0'].values
i4 = df['Unnamed: 0'].values
i5 = df['Unnamed: 0'].values
i6 = df['Unnamed: 0'].values

In [5]:
# files to search
s1 = h51[bd1.astype(str)=='nan']
s2 = h52[bd2.astype(str)=='nan']
s3 = h53[bd3.astype(str)=='nan']
s4 = h54[bd4.astype(str)=='nan']
s5 = h55[bd5.astype(str)=='nan']
s6 = h56[bd6.astype(str)=='nan']

si1 = i1[bd1.astype(str)=='nan']
si2 = i2[bd2.astype(str)=='nan']
si3 = i3[bd3.astype(str)=='nan']
si4 = i4[bd4.astype(str)=='nan']
si5 = i5[bd5.astype(str)=='nan']
si6 = i6[bd6.astype(str)=='nan']

hi1 = [1 for i in si1]
hi2 = [2 for i in si2]
hi3 = [3 for i in si3]
hi4 = [4 for i in si4]
hi5 = [5 for i in si5]
hi6 = [6 for i in si6]

s_all = np.concatenate([s1, s2, s3, s4, s5, s6])
si_all = np.concatenate([si1, si2, si3, si4, si5, si6])
hi_all = np.concatenate([hi1, hi2, hi3, hi4, hi5, hi6])
inp = np.transpose([si_all, hi_all, s_all])
print(inp)
print(len(s1), len(s2), len(s3), len(s4), len(s5), len(s6))

[[6231 1
  '/datag/pipeline/AGBT18A_999_70/collate2/spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58265_43116_HIP116495_0021.gpuspec.0000.h5']
 [6279 1
  '/datag/pipeline/AGBT16B_999_48/holding/spliced_blc0001020304050607_guppi_57680_08933_HIP116584_0021.gpuspec.0000.h5']
 [6280 1
  '/datag/pipeline/AGBT17A_999_18/holding/spliced_blc0001020304050607_guppi_57815_64506_HIP116584_0050.gpuspec.0000.h5']
 ...
 [28671 6
  '/datag/pipeline/AGBT18B_999_08/blp15/blc15_guppi_58350_01831_HIP71875_0055.gpuspec.0000.h5']
 [39112 6
  '/datag/pipeline/AGBT17A_999_74/holding/spliced_blc0001020304050607_guppi_57903_52843_HIP13027_0019.gpuspec.0000.h5']
 [39113 6
  '/datag/pipeline/AGBT18A_999_07/collate0/spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58165_86394_HIP13027_0015.gpuspec.0000.h5']]
106 111 110 110 112 117


In [11]:
print(np.where(s_all=='/datag/pipeline/AGBT23B_999_23/blc11_blp11/blc11_guppi_60306_38062_HIP38674_0118.rawspec.0000.h5'))

(array([331]),)


In [ ]:
### FOR BLISS:

outdir = '/datax/scratch/benjb/bl_nearby_stars/bliss_final_100_search_063026/'
logdir = '/datax/scratch/benjb/bl_nearby_stars/bliss_logs/'

logger = logging.getLogger(__name__)

def process_files(xxx):
    n = xxx[0]
    sbatch = xxx[1]
    snr = 20

    # Remove all handlers associated with the root logger object.
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=f'{logdir}0_3_6_{n}_final100_070726.log', filemode="a", level=logging.DEBUG)

    logger.info('Beginning.')

    for i in range(len(sbatch)):

        stopscan = [95,95,95,51,61,94,95]
        if int(n) < 3:
            break
        if i < stopscan[int(n)]:
            continue

        index = sbatch[i,0]

        test_file = sbatch[i,2]
        if not os.path.exists(test_file):
            logger.info(f'{i}: .h5 file not found: {test_file}')
            continue
        fb = bl.Waterfall(test_file, load_data=False)
        nfc = fb.header['nchans'] # number of fine channels

        # check for configuration
        if nfc % (2**20) == 0:
            # configuration is normal
            ncc = nfc // 2**20
            nfpc = 2**20
            config = 'u' # usual
            pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response.f32'
        elif nfc % (1033216) == 0:
            ncc = nfc // 1033216
            nfpc = 1033216
            config = 'o' # old
            pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response_1033216.f32'
        else:
            print('Unusual configuration; skipping for now.')
            continue
        h5idx = sbatch[i,1]
        file = test_file
        # if not 'spliced' in file:
        #     logger.info('  Unspliced file! Continuing ...')
        #     continue
        logger.info(f'Searching #{i} of {len(sbatch)} files (ID #{index}, file #{h5idx} of 6 in its cadence: {file}) ...')

        # run BLISS for each coarse channel
        for k in range(ncc):
            # check whether dat already exists:
            datcheck = outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
            if os.path.exists(datcheck):
                with open(datcheck) as f:
                    lines = f.readlines()
                if len(lines) > 0:
                    if k % 200 == 0:
                        logger.info(f'  Channel {k} of {ncc}: Already searched!')
                else:
                    logger.info(f'  Channel {k} of {ncc}: Empty .dat. Re-searching ...')
                    try: # for spliced files, do one cc at a time
                        if k%200 == 0:
                            logger.info(f'  Channel {k} of {ncc}: Searching ...')
                        console = f'bliss_find_hits {file} -e {pfb} -d cuda:0 -md -4 -MD 4 -s {snr} --number-coarse 1 -c {k} --nchan-per-coarse {nfpc} --distance 30 --output ' + outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                        os.system(console)
                    except:
                        print('Search failed; check later.')
            if not os.path.exists(datcheck):
                try: # for spliced files, do one cc at a time
                    if k%200 == 0:
                        logger.info(f'  Channel {k} of {ncc}: Searching ...')
                    console = f'bliss_find_hits {file} -e {pfb} -d cuda:0 -md -4 -MD 4 -s {snr} --number-coarse 1 -c {k} --nchan-per-coarse {nfpc} --distance 30 --output ' + outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                    os.system(console)
                except:
                    print('Search failed; check later.')
            else:
                if k % 200 == 0:
                    logger.info(f'  Channel {k} of {ncc}: Already searched!')
    logger.info('Done!')


if __name__ == "__main__":
    # Define n sets of files
    nsplit = 7
    batch_size = len(inp) // nsplit

    file_sets = [[str(i), inp[i*batch_size:(i+1)*batch_size]] for i in range(nsplit-1)]
    file_sets.append([str(nsplit-1), df.iloc[(nsplit-1)*batch_size:len(inp)]])

    #file_sets = [[str(i), df.iloc[i*batch_size+len(df)*2//3:(i+1)*batch_size+len(df)*2//3]] for i in range(nnodes-1)] # leave a GPU open
    #file_sets = [[str(i), df.iloc[i*batch_size:(i+1)*batch_size]] for i in range(nnodes)]
    #file_sets.append([str(nnodes), df.iloc[nnodes*batch_size+len(df)*3//4:len(df)]])

    # Create a pool of n processes
    with multiprocessing.Pool(processes=4) as pool:
        # Map the process_files function to each file set
        pool.map(process_files, file_sets[:4])

    print("All files processed.")